<a href="https://colab.research.google.com/github/hmmnyamminji/DL/blob/main/day06_practice3_WnB_%EC%8B%A4%ED%97%98%EC%B6%94%EC%A0%81_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.8 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.42.1
    Uninstalling opentelemetry-api-1.42.1:
      Successfully uninstalled opentelemetry-api-1.42.1
  Attempting uninstall: wandb
    Found existing installation: wandb 0.28.1
    Uninstalling wandb-0.28.1:
      Successfully uninstalled wandb-0.28.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-semantic-conventions 0.63b1 requires opentelemetry-api==1.42.1, but you have opentelemetry-api 1.44.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.44.0 which is incompatible.
opentelemetry-sdk 1.42.1 requires opentelemetry-api==1.42.

In [2]:
import os
os.environ["WANDB_MODE"] = "online"
import wandb
print("WANDB_MODE =",  os.environ["WANDB_MODE"])

WANDB_MODE = online


In [3]:
wandb.login() # 로그인 (방법 A: 직접 입력)

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: gimm00999 (gimm00999-kwu) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
# 로그인 (방법 B: Colab Service - 권장 방법)
import os
from google.colab import userdata
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")

import wandb
wandb.login()   # 키가 환경변수에 있으면 입력창 없이 바로 로그인

True

In [5]:
# 피마 인디언 당뇨 예측
# 'CSV -> 판다스 탐색 -> 텐서 -> Dataset/DataLoader -> 학습' 전체 파이프라인

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"

In [6]:
# 셀 1. 재현성 - 시드 고정은 실험 기록의 전체


In [7]:
# 셀 2. 데이터 불러오
CSV_URL = "https://raw.githubusercontent.com/taehojo/deeplearning_4th/master/data/pima-indians-diabetes3.csv"
CSV_PATH = "pima-indians-diabetes3.csv"

def load_pima():
  import os
  if os.path.exists(CSV_PATH):
    print(f"로컬 파일 재사용: {CSV_PATH}")
    return pd.read_csv(CSV_PATH)

  try:
    df = pd.read_csv(CSV_URL)
    df.to_csv(CSV_PATH, index=False)
    print(f"다운로드 완료 -> {CSV_PATH} 저장 (다음 실행부턴 재사용)")
    return df
  except Exception as e:
    raise SystemExit(
        f"데이터 다운로드 실패: {e}\n"
    )

df = load_pima()
print("데이터 크기:", df.shape)
print(df.head())

다운로드 완료 -> pima-indians-diabetes3.csv 저장 (다음 실행부턴 재사용)
데이터 크기: (768, 9)
   pregnant  plasma  pressure  thickness  insulin   bmi  pedigree  age  \
0         6     148        72         35        0  33.6     0.627   50   
1         1      85        66         29        0  26.6     0.351   31   
2         8     183        64          0        0  23.3     0.672   32   
3         1      89        66         23       94  28.1     0.167   21   
4         0     137        40         35      168  43.1     2.288   33   

   diabetes  
0         1  
1         0  
2         1  
3         0  
4         1  


In [8]:
X, y = df.iloc[:, :-1].values, df.iloc[:, -1].values

X_tr, X_tmp, y_tr, y_tmp = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)
X_val, _, y_val, _ = train_test_split(
    X_tmp, y_tmp, test_size=0.5, random_state=42
)

scaler =StandardScaler()
def to_t(Xa, ya, fit=False):
  Xs = scaler.fit_transform(Xa) if fit else scaler.transform(Xa)
  return (
      torch.tensor(Xs, dtype=torch.float32).to(device),
      torch.tensor(ya, dtype=torch.float32).reshape(-1, 1).to(device)
  )

X_tr_t, y_tr_t = to_t(X_tr, y_tr, fit=True)
X_val_t, y_val_t = to_t(X_val, y_val)

print(f"학습 {len(X_tr)} / 검증 {len(X_val)}")

학습 460 / 검증 154


In [9]:
# 셀 3. 데이터로더 + 모델 정의
from torch.utils.data import DataLoader, TensorDataset

train_loader = DataLoader(
    TensorDataset(X_tr_t.cpu(), y_tr_t.cpu()),
    batch_size=32, shuffle=True,
    generator=torch.Generator().manual_seed(42)) # 셔플 난수도 시드 고정

def make_model():
  return nn.Sequential(
      nn.Linear(8, 64), nn.ReLU(),
      nn.Linear(64, 32), nn.ReLU(),
      nn.Linear(32, 1), nn.Sigmoid())

In [13]:
# 배치 정규화 (Batch Nomarlization)
#def make_model():
#      nn.Linear(8, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.3),
#      nn.Linear(64, 32), nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(0.3),
#      nn.Linear(32, 1), nn.Sigmoid())

# BatchNorm1d는 배치 단위로 평균/분산을 계산
# 학습 시: 현재 배치의 평균/분산 사용
# 평가 시: 학습 중 누적된 running 평균/분산 사용
# 열은 망 + 작은 표 데이터에서 효과가 미미

In [10]:
# 셀 4. WAB 패턴 - init(시작) -> log(기록) -> finish(마침) (학습률 3종 비교)

for lr in [0.05, 0.005, 0.0005]: # 학습률 3종 비교

    run = wandb.init(
        project="dt-day06-training-rate",
        name=f"lr_{lr}",
        config={"lr": lr, "epochs": 40, "batch_size": 32, "seed": 42},
        reinit="finish_previous"
    )

    torch.manual_seed(42) # lr 안 바꾸고 시드는 고정 (공정 비교)
    model = make_model().to(device)
    loss_fn = nn.BCELoss()
    opt = torch.optim.Adam(model.parameters(), lr=run.config.lr)

    for epoch in range(40):
        model.train()

        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            loss = loss_fn(model(xb), yb)

            opt.zero_grad()
            loss.backward()
            opt.step()

        model.eval()

        with torch.no_grad():
            val_pred = model(X_val_t)
            val_loss = loss_fn(val_pred, y_val_t).item()
            val_acc = ((val_pred > 0.5) == y_val_t.bool()).float().mean().item()

        wandb.log({"val_loss": val_loss, "val_acc": val_acc}) # 2. log에 에폭 지표 기록 -> 웹 차트

    wandb.finish() # 3. finish: 이 run을 닫음 (다음 run과 안 엉키게)

    print(f"[run] lr ={lr:<7} 최종 val loss {val_loss:.3f} | val acc {val_acc:.3f}")

val_acc,▁██▇█▇▄▇▇▅▇▅▅▆▅▆▅▁▁▄▅▅▅▇▅▂▁▆▄▇▅▅▇▇▅▄▇▆▆█
val_loss,▁▁▁▁▁▁▁▁▁▁▁▃▂▂▁▃▃▂▄▃▆▂▃▃▃▅▂▃▂▅▂▇▄█▆▄▂█▇█
val_acc,0.76623
val_loss,4.31589


[run] lr =0.05    최종 val loss 4.316 | val acc 0.766


val_acc,▄▅██▆▅▆▆▇▇▅▆▄▅▂▅▄▄▄▄▃▃▄▃▄▃▄▃▄▄▂▃▃▂▄▄▃▃▃▁
val_loss,▂▁▁▁▁▁▁▁▁▁▁▂▂▂▂▂▂▂▃▃▃▄▃▃▄▄▄▄▅▅▅▅▆▆▇▆▇█▇█
val_acc,0.69481
val_loss,0.83646


[run] lr =0.005   최종 val loss 0.836 | val acc 0.695


val_acc,▁▁▁▁▂▅▆▅▆▇█▇▇▇▇▇▇▇▇▇▇▇▇▇▇▆▇▇▆▇▆▇▇▇▇▇▇▇▇▇
val_loss,█▇▆▆▅▄▄▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_acc,0.75974
val_loss,0.4568


[run] lr =0.0005  최종 val loss 0.457 | val acc 0.760
